# 🔬 XAI-SkinNet — Full Training Notebook (Google Colab)

**Architecture:** EfficientNet-B4 + ViT-B/16 hybrid with attention-gated fusion + Grad-CAM + SHAP  
**Dataset:** HAM10000 (10,015 dermoscopic images, 7 classes)  
**Authors:** Suhani Singhania, Kashvi Agrawal, Riya Jain — VIT Vellore

---
## 📋 Before You Begin

1. **Runtime → Change runtime type → GPU (T4)** — do this first!
2. You need the **5 HAM10000 files** from Kaggle:
   - `HAM10000_metadata.csv`
   - `HAM10000_images_part_1.zip` (or unzipped folder)
   - `HAM10000_images_part_2.zip` (or unzipped folder)
3. Choose **one** data loading method in Step 3 below.

---

## ✅ Step 1 — Check GPU

In [ ]:
import torch
print('=== GPU Check ===')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → GPU')

## ✅ Step 2 — Clone Repo & Install Dependencies

In [ ]:
import os
# ← UPDATE THIS with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/xai-skinnet-main.git'

if not os.path.exists('/content/xai-skinnet-main'):
    !git clone {REPO_URL} /content/xai-skinnet-main
else:
    print('Repo found — pulling latest...')
    !git -C /content/xai-skinnet-main pull

%cd /content/xai-skinnet-main
print('Working dir:', os.getcwd())

In [ ]:
!pip install -q \
    'timm>=0.9.12' 'scikit-learn>=1.3' 'imbalanced-learn>=0.11' \
    'grad-cam>=1.5.0' 'shap>=0.45' 'kagglehub>=0.3' \
    'kaggle>=1.6' 'seaborn>=0.12' 'tqdm>=4.66'
print('✅ Dependencies installed.')

## ✅ Step 3 — Load HAM10000 Dataset

**Run only ONE of the three methods below.**

### 📂 Method A — Upload from your PC
Upload: `HAM10000_metadata.csv`, `HAM10000_images_part_1.zip`, `HAM10000_images_part_2.zip`

In [ ]:
import os, zipfile
from google.colab import files

DATA_ROOT = '/content/xai-skinnet-main/data/raw'
os.makedirs(DATA_ROOT, exist_ok=True)

print('📤 Select your HAM10000 files (CSV + zip archives)...')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = os.path.join(DATA_ROOT, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')
    if fname.endswith('.zip'):
        print(f'  Extracting {fname}...')
        with zipfile.ZipFile(dest, 'r') as zf:
            zf.extractall(DATA_ROOT)
        print(f'  ✅ Extracted')

print('\n📁 data/raw contents:')
for item in sorted(os.listdir(DATA_ROOT)):
    print(f'  {item}')

### ☁️ Method B — Google Drive
Use if you uploaded the HAM10000 files to Drive.

In [ ]:
import os, zipfile, shutil
from google.colab import drive
drive.mount('/content/drive')

# ← UPDATE: path in your Drive containing the HAM10000 files
DRIVE_FOLDER = '/content/drive/MyDrive/HAM10000'
DATA_ROOT    = '/content/xai-skinnet-main/data/raw'
os.makedirs(DATA_ROOT, exist_ok=True)

for item in os.listdir(DRIVE_FOLDER):
    src = os.path.join(DRIVE_FOLDER, item)
    dst = os.path.join(DATA_ROOT, item)
    if not os.path.exists(dst):
        print(f'Copying {item}...')
        shutil.copy2(src, dst) if os.path.isfile(src) else shutil.copytree(src, dst)
    if item.endswith('.zip'):
        print(f'Extracting {item}...')
        with zipfile.ZipFile(dst, 'r') as zf:
            zf.extractall(DATA_ROOT)

print('\n📁 data/raw:', sorted(os.listdir(DATA_ROOT)))

### 🏆 Method C — Kaggle API (auto-download)
Use if you have a `kaggle.json` API key.

In [ ]:
import os
from google.colab import files
print('Upload your kaggle.json:')
files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

DATA_ROOT = '/content/xai-skinnet-main/data/raw'
os.makedirs(DATA_ROOT, exist_ok=True)
!python /content/xai-skinnet-main/data/download_data.py --dataset ham10000 --out {DATA_ROOT}
print('\n📁 data/raw:', sorted(os.listdir(DATA_ROOT)))

## ✅ Step 4 — Verify Dataset

In [ ]:
import pandas as pd
from pathlib import Path

DATA_ROOT = '/content/xai-skinnet-main/data/raw'
print('=== Dataset Verification ===')

csv_hits = list(Path(DATA_ROOT).rglob('HAM10000_metadata.csv'))
if csv_hits:
    df = pd.read_csv(csv_hits[0])
    print(f'✅ Metadata CSV: {len(df)} rows')
    print('   Class distribution:')
    print(df['dx'].value_counts().to_string())
else:
    print('❌ metadata CSV not found!')

for part in ['HAM10000_images_part_1', 'HAM10000_images_part_2']:
    hits = list(Path(DATA_ROOT).rglob(part))
    if hits:
        n = len(list(hits[0].glob('*.jpg')))
        print(f'✅ {part}: {n} images')
    else:
        n_total = len(list(Path(DATA_ROOT).rglob('*.jpg')))
        print(f'⚠️  {part} not found. Total JPGs found: {n_total}')

## ✅ Step 5 — Configure Hyperparameters

In [ ]:
import yaml
CONFIG_PATH = '/content/xai-skinnet-main/configs/config.yaml'

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── Edit these ────────────────────────────────────────────────────────────
cfg['train']['epochs']              = 60    # use 3-5 for a quick smoke test
cfg['train']['batch_size']          = 16    # 16 safe on T4; 32 if you have VRAM
cfg['train']['lr']                  = 2.0e-4
cfg['train']['freeze_epochs']       = 10
cfg['train']['early_stop_patience'] = 10
cfg['train']['num_workers']         = 2
cfg['model']['use_metadata']        = False  # True = multimodal extension

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('=== Active Config ===')
print(yaml.dump(cfg, default_flow_style=False))

## 🚀 Step 6 — Train!

T4 GPU estimate: ~2-4 hours for 60 epochs. Set `epochs=3` in Step 5 for a smoke test first.

In [ ]:
import os
os.chdir('/content/xai-skinnet-main')
DATA_ROOT = '/content/xai-skinnet-main/data/raw'

!python -m src.train --config configs/config.yaml --data-root {DATA_ROOT}

## 📊 Step 7 — Evaluate

In [ ]:
import os
os.chdir('/content/xai-skinnet-main')
RUN_NAME  = 'xai_skinnet_v1'
DATA_ROOT = '/content/xai-skinnet-main/data/raw'
CKPT      = f'runs/{RUN_NAME}/best.pt'

!python -m src.evaluate --config configs/config.yaml --checkpoint {CKPT} --data-root {DATA_ROOT}

## 🌡️ Step 8 — Temperature Calibration

In [ ]:
import os
os.chdir('/content/xai-skinnet-main')
RUN_NAME  = 'xai_skinnet_v1'
DATA_ROOT = '/content/xai-skinnet-main/data/raw'
CKPT      = f'runs/{RUN_NAME}/best.pt'

!python -m src.calibrate --checkpoint {CKPT} --data-root {DATA_ROOT}
print('\n--- Re-evaluation with calibration ---')
!python -m src.evaluate --config configs/config.yaml --checkpoint {CKPT} --data-root {DATA_ROOT}

## 🔍 Step 9 — Grad-CAM Heatmap

In [ ]:
import os
from pathlib import Path
from IPython.display import Image as IPImg, display
os.chdir('/content/xai-skinnet-main')

RUN_NAME  = 'xai_skinnet_v1'
DATA_ROOT = '/content/xai-skinnet-main/data/raw'
CKPT      = f'runs/{RUN_NAME}/best.pt'

jpgs = list(Path(DATA_ROOT).rglob('*.jpg'))
if jpgs:
    SAMPLE = str(jpgs[0])
    print(f'Sample: {SAMPLE}')
    !python -m src.explainability.gradcam --checkpoint {CKPT} --image {SAMPLE}
    heatmaps = list(Path('.').glob('gradcam_*.jpg'))
    if heatmaps:
        display(IPImg(str(heatmaps[-1])))
else:
    print('No images found — run Step 3 first.')

## 🔍 Step 10 — SHAP Attribution

In [ ]:
import os
os.chdir('/content/xai-skinnet-main')
RUN_NAME  = 'xai_skinnet_v1'
DATA_ROOT = '/content/xai-skinnet-main/data/raw'
CKPT      = f'runs/{RUN_NAME}/best.pt'

!python -m src.explainability.shap_explain --checkpoint {CKPT} --data-root {DATA_ROOT} --n-background 50

## 📈 Step 11 — Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path

RUN_NAME     = 'xai_skinnet_v1'
history_path = Path(f'/content/xai-skinnet-main/runs/{RUN_NAME}/history.json')

if not history_path.exists():
    print('Run Step 6 first.')
else:
    history    = json.loads(history_path.read_text())
    epochs     = [h['epoch']      for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss   = [h['val_loss']   for h in history]
    train_acc  = [h['train_acc']  for h in history]
    val_acc    = [h['val_acc']    for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, train_loss, 'b-o', ms=3, label='Train'); ax1.plot(epochs, val_loss,  'r-o', ms=3, label='Val')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, [a*100 for a in train_acc], 'b-o', ms=3, label='Train'); ax2.plot(epochs, [a*100 for a in val_acc], 'r-o', ms=3, label='Val')
    ax2.set(title='Accuracy', xlabel='Epoch', ylabel='Acc %'); ax2.legend(); ax2.grid(alpha=0.3)

    best = max(history, key=lambda h: h['val_acc'])
    fig.suptitle(f'XAI-SkinNet — Best val_acc={best["val_acc"]*100:.2f}% (ep {best["epoch"]})', fontweight='bold')
    plt.tight_layout()
    out = f'/content/xai-skinnet-main/runs/{RUN_NAME}/training_curves.png'
    plt.savefig(out, dpi=150); plt.show()
    print(f'Saved: {out}')

## 💾 Step 12 — Save to Google Drive  ⚠️ Run this before closing Colab!

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RUN_NAME = 'xai_skinnet_v1'
SRC      = f'/content/xai-skinnet-main/runs/{RUN_NAME}'
DST      = f'/content/drive/MyDrive/XAI_SkinNet_Results/{RUN_NAME}'
os.makedirs(DST, exist_ok=True)

for f in ['best.pt','history.json','training_curves.png','test_split.csv','val_split.csv','temperature.json']:
    s = os.path.join(SRC, f)
    if os.path.exists(s):
        shutil.copy2(s, os.path.join(DST, f))
        print(f'✅ {f} → Drive')

print(f'\n✅ All results in: {DST}')

## 🔬 Step 13 — XAI Faithfulness (optional)

In [ ]:
import os
os.chdir('/content/xai-skinnet-main')
RUN_NAME  = 'xai_skinnet_v1'
DATA_ROOT = '/content/xai-skinnet-main/data/raw'
CKPT      = f'runs/{RUN_NAME}/best.pt'
TEST_CSV  = f'runs/{RUN_NAME}/test_split.csv'

!python -m src.explainability.xai_eval \
    --checkpoint {CKPT} --data-root {DATA_ROOT} --test-csv {TEST_CSV} --n-samples 30

---
## 📋 Quick Reference

| Step | What it does |
|------|--------------|
| 1 | Check GPU |
| 2 | Clone repo + install deps |
| 3 | Load HAM10000 (Method A: upload / B: Drive / C: Kaggle API) |
| 4 | Verify dataset |
| 5 | Configure hyperparameters |
| **6** | **🚀 Train the model** |
| 7 | Evaluate (acc / F1 / AUC / ECE + confusion matrix) |
| 8 | Temperature calibration |
| 9 | Grad-CAM heatmap |
| 10 | SHAP attribution |
| 11 | Plot training curves |
| **12** | **💾 Save to Drive (important!)** |
| 13 | XAI faithfulness eval |

**7 Classes:** MEL · NV · BCC · AKIEC · BKL · DF · VASC